### CSE5026 Foundation and Frontiers of Cognitive Science
## Lab: Context-Free Grammars and Parsing


In [ ]:
from collections import defaultdict
from typing import List, Dict, Tuple
from pprint import pprint
import io

## T1. Examine Grammar Rules

Open the `atis3.cfg` file and examine its content. 

The start symbol is `TOP`, defined below `# Startsymbols` at the beginning of the file.

Lines after the comment line "# Phrasal rules" and before "# Lexical rules" are rules for non-terminal symbols, for example: `NP -> NP PP`. 

Lines after the comment line "# Lexical rules" are rules for terminal symbols, for example: `NP -> aircraft`.

**Task**: Count the number of rules whose left-hand side symbol is `NP` (including both phrasal and lexical rules).


In [ ]:
NP_count = 0

with open('atis3.cfg', 'r') as f:
    ### START YOUR CODE ###
    pass
    ### END YOUR CODE ###

# Test result
print('NP rules count:', NP_count)

# You should expect to see:
# NP rules count: 276

## T2. Parse Grammar Rules

Define `parse_rule` to read one grammar rule and return `(lhs, rhs)`, where `lhs` is a string and `rhs` is a tuple of strings.

**Note**:
- The left hand side is a `str` and the right hand side is a `tuple` of `str`. Thus, in cases of lexical rules, the right-hand side tuple will have only one element.

In [ ]:
def parse_rule(rule: str) -> Tuple[str, Tuple[str]]:
    ### START YOUR CODE ###
    ### END YOUR CODE ###
    return lhs, rhs

# Test result
rule1 = 'NP -> DET N'
print('rule:', rule1)
lhs, rhs = parse_rule(rule1)
print('lhs:', lhs)
print('rhs:', rhs)

print()
rule2 = 'NP -> aircraft'
print('rule:', rule2)
lhs, rhs = parse_rule(rule2)
print('lhs:', lhs)
print('rhs:', rhs)

# You should expect to see:
# rule: NP -> DET N
# lhs: NP
# rhs: ('DET', 'N')

# rule: NP -> aircraft
# lhs: NP
# rhs: ('aircraft',)

**Next**, integrate the above functions to the `CFG` class. 

**Notes**:
- In the class method `read_rules`, read the rules line by line and parse them using the `parse_rule` function.
- The class member `lhs_to_rules` maps each left-hand side symbol (`str`) to a list of complete `(lhs, rhs)` rules returned by `parse_rule`.
- Similarly, `rhs_to_rules` maps each right-hand side (`tuple`) to a list of complete rules.

In [ ]:
class CFG(object):
    def __init__(self, grammar_file: io.TextIOWrapper):
        self.rhs_to_rules = defaultdict(list)
        self.lhs_to_rules = defaultdict(list)
        self.startsymbol = None
        self.read_rules(grammar_file)

    def read_rules(self, grammar_file: io.TextIOWrapper):
        for line in grammar_file:
            line = line.strip()
            if line and not line.startswith("#"):
                if "->" in line:
                    ### START YOUR CODE ###
                    pass
                    ### END YOUR CODE ###
                else:
                    startsymbol = line.strip()
                    self.startsymbol = startsymbol
    
    def parse_rule(self, rule: str) -> Tuple[str, Tuple[str]]:
        ### START YOUR CODE ###
        # Copy from the previous cell
        ### END YOUR CODE ###
        return lhs, rhs

In [ ]:
# Test result
with open('atis3.cfg', 'r') as f:
    cfg = CFG(f)
    print('rhs_to_rules:', len(cfg.rhs_to_rules))
    print('lhs_to_rules:', len(cfg.lhs_to_rules))
    print('startsymbol:', cfg.startsymbol)

    print()
    print('# of "NP -> *":', len(cfg.lhs_to_rules['NP']))
    print('# of "* -> aircraft":', len(cfg.rhs_to_rules[('aircraft',)]))

    print()
    print('all rules for "* -> aircraft":')
    for rule in cfg.rhs_to_rules[('aircraft',)]:
        print(rule)

# You should expect to see:
# rhs_to_rules: 852
# lhs_to_rules: 300
# startsymbol: TOP
# # of "NP -> *": 276
# # of "* -> aircraft": 2
# all rules for "* -> aircraft":
# ('AIRCRAFT', ('aircraft',))
# ('NP', ('aircraft',))

You can see that `rhs_to_rules` provides a convenient way to find all rules that have a specific right-hand side symbol.

---

## T3. CKY Recognition

We first implement the CKY recognition algorithm cell by cell.

**Note**
- First, the **super-diagonal** elements (directly above the diagonal elements), i.e., `table[i][i+1]`, correspond to the span of length 1 in the input sentence.
- We can retrieve their rules from the `rhs_to_rules` dictionary, using the terminal symbol as the key (in tuple of length 1).
- `rhs_to_rules[rhs]` returns a list of `(lhs, rhs)` rules. Append the `lhs` of each rule to `table[i][i+1]`.

In [ ]:
tokens = 'flights from miami to cleveland .'.split()
n = len(tokens)

# Initialize the table
table = [[[] for _ in range(n+1)] for _ in range(n+1)]

# Fill the super-diagonal elements
for j in range(n):
    ### START YOUR CODE ###
    pass
    ### END YOUR CODE ###


# Test result
pprint(table)

# You should expect to see:
# [[[], ['FLIGHTS', 'NP'], [], [], [], [], []],
#  [[], [], ['FROM', 'PP'], [], [], [], []],
#  [[], [], [], ['NP'], [], [], []],
#  [[], [], [], [], ['TO', 'X'], [], []],
#  [[], [], [], [], [], ['CLEVELAND', 'NP'], []],
#  [[], [], [], [], [], [], ['PUN']],
#  [[], [], [], [], [], [], []]]

**Next**, go through each cell `[i,j]` in the rest of the table, and find all non-terminal symbols.

**Note**
- Each table cell `[i,j]` is initialized as an empty list. 
- Go through each `(rhs, rules)` item in `rhs_to_rules`. For binary rules (`len(rhs) == 2`), check whether `rhs[0]` is in `table[i][k]` and `rhs[1]` is in `table[k][j]` for some `k` between `i+1` and `j-1`.
  If so, append the `lhs` of each matching rule to `table[i][j]`.
- For simplicity, each cell stores unique non-terminal symbols. Their order after deduplication may differ from the example output.

In [ ]:
# Fill the rest of the table

for j in range(2, n+1):
    for i in range(j-2, -1, -1):
        for k in range(i+1, j):
            ### START YOUR CODE ###
            pass
            ### END YOUR CODE ###
        table[i][j] = list(set(table[i][j]))

In [ ]:
# Test result
pprint(table[0])

# You should see the following output:
# [[],
#  ['FLIGHTS', 'NP'],
#  ['NP', 'SQBAR', 'VPBAR', 'FRAG'],
#  ['NPBAR', 'FRAGBAR', 'NP', 'SQBAR', 'VPBAR', 'FRAG'],
#  [],
#  ['NPBAR', 'FRAGBAR', 'NP', 'SQBAR', 'VPBAR', 'FRAG'],
#  ['TOP']]

**Lastly**, after the table is filled, we can check if the start symbol is in cell `[0,n]`. 

If so, the input sentence is recognized by the grammar, i.e., grammatical; otherwise not.

In [ ]:
if cfg.startsymbol in table[0][n]:
    print('The input string is grammatical')
else:
    print('The input string is not grammatical')

# You should see the following output:
# The input string is grammatical

In this case, "flights from miami to cleveland ." is grammatical. 

You can try scrambling the sentence to see if the output changes.

---

## T4. Integrate into the CKY class

Integrate the above code to the `is_grammatical` method of `CKY` class, which takes a list of tokens as input and returns a boolean value.

In [ ]:
class CKY(object):
    def __init__(self, cfg: CFG):
        self.cfg = cfg

    def is_grammatical(self, tokens: List[str]) -> bool:
        ### START YOUR CODE ###
        pass
        ### END YOUR CODE ###

In [ ]:
# Test CKY class
with open('atis3.cfg', 'r') as f:
    cfg = CFG(f)
    cky = CKY(cfg)
    print(cky.is_grammatical('flights from miami to cleveland .'.split()))
    print(cky.is_grammatical('miami flights cleveland from to .'.split()))

---

## From Recognition to Parsing

T1–T4 implement CKY recognition.

In T5–T6, extend the CKY algorithm to reconstruct one tree and enumerate all different trees.

Submit the completed notebook file with all tasks done on Blackboard. 

See [README_en.md](README_en.md) for the full submission requirements.


## T5. Add All Backpointers to the Table

Implement `build_backpointer_table(tokens, cfg)` using your CKY recognition code. 

Return a dictionary whose entry `table[(i, j)]` stores the non-terminal symbols for the half-open span `[i, j)`, together with **all distinct backpointers** for each symbol.

One possible record layout is:
- Lexical entry: `table[(i, i+1)][A] = [word]`.
- Binary entry: `table[(i, j)][A] = [((B, i, k), (C, k, j)), ...]`.

Each binary record describes one grammar rule using `A -> B C`; the two child records point to smaller spans. 

You may design a different record layout, provided T5 and T6 use it consistently.

Use only binary phrasal rules and lexical rules, as in `atis3.cfg`. Implement the algorithm yourself using the Python standard library.


In [ ]:
def build_backpointer_table(tokens: List[str], cfg: CFG) -> Dict:
    """Build a CKY table containing all distinct backpointers."""
    ### START YOUR CODE ###
    raise NotImplementedError("Complete T5 before running the tests.")
    ### END YOUR CODE ###


In [ ]:
# T5: show the root cell and one internal cell for the example sentence.
tokens = 'flights from miami to cleveland .'.split()
n = len(tokens)
table2 = build_backpointer_table(tokens, cfg)

print('Root cell [0, n):')
pprint(table2[(0, n)])
print('Internal cell [0, n-1):')
pprint(table2[(0, n - 1)])
assert cfg.startsymbol in table2[(0, n)]
assert table2[(0, n - 1)]

## T6. Build One Tree and All Trees Recursively

Implement both functions below using the code from T5. 

A tree is a nested tuple:
- Lexical node: `(symbol, word)`.
- Binary node: `(symbol, left_tree, right_tree)`.

`build_tree(table, i, j, symbol)` returns any one complete tree rooted at `symbol` for `[i, j)`, or `None` if this node is absent.

`build_tree_full(table, i, j, symbol)` returns a **list of all different trees**, or `[]` when the node is absent. 

Traverse every backpointer and combine **every left subtree with every right subtree**. 
Note that returning only one tree per split is insufficient if a child span is ambiguous.

Handle missing roots and empty input without raising an error. Trees are compared by their nested tuple structure; their output order does not matter.


In [ ]:
def build_tree(table, i, j, symbol):
    """Return one nested-tuple tree, or None if the node is absent."""
    ### START YOUR CODE ###
    raise NotImplementedError("Complete T6: build_tree before running the tests.")
    ### END YOUR CODE ###


In [ ]:
def build_tree_full(table, i, j, symbol):
    """Return all distinct nested-tuple trees, or [] if the node is absent."""
    ### START YOUR CODE ###
    raise NotImplementedError("Complete T6: build_tree_full before running the tests.")
    ### END YOUR CODE ###


### Required Tests

Keep the following tests and their actual outputs. The helpers below check leaf order and verify that every node uses a rule in the supplied grammar; they do not implement parsing.

Complete T1–T6 before running these tests. Passing only the positive example does not demonstrate correct ambiguity handling.


In [ ]:
# Provided validation helpers: keep these checks in your submission.
def tree_leaves(tree):
    if len(tree) == 2:
        return [tree[1]]
    return tree_leaves(tree[1]) + tree_leaves(tree[2])


def assert_valid_tree(tree, grammar):
    assert isinstance(tree, tuple) and len(tree) in (2, 3)
    symbol = tree[0]
    if len(tree) == 2:
        assert isinstance(tree[1], str)
        rhs = (tree[1],)
    else:
        for child in tree[1:]:
            assert_valid_tree(child, grammar)
        rhs = (tree[1][0], tree[2][0])
    assert (symbol, rhs) in grammar.lhs_to_rules.get(symbol, [])


def assert_tree_set(trees, grammar, tokens):
    assert isinstance(trees, list) and trees
    assert len(trees) == len(set(trees)), 'Duplicate trees found.'
    for tree in trees:
        assert tree[0] == grammar.startsymbol
        assert tree_leaves(tree) == tokens
        assert_valid_tree(tree, grammar)


In [ ]:
# 1. Positive example: inspect one tree and validate every generated tree.
assert cky.is_grammatical(tokens) is True
one_tree = build_tree(table2, 0, n, cfg.startsymbol)
all_trees = build_tree_full(table2, 0, n, cfg.startsymbol)
assert_tree_set(all_trees, cfg, tokens)
assert one_tree in all_trees

print('One parse tree:')
pprint(one_tree)
print('Number of distinct parse trees:', len(all_trees))
print('All roots, leaves and grammar productions checked.')


In [ ]:
# 2–4. Rejected inputs: recognition is False and no tree is returned.
rejected_inputs = {
    'scrambled': 'miami flights cleveland from to .'.split(),
    'unknown word': 'flights from unknown_city to cleveland .'.split(),
    'empty input': [],
}
assert not cfg.rhs_to_rules.get(('unknown_city',), [])
for label, bad_tokens in rejected_inputs.items():
    bad_table = build_backpointer_table(bad_tokens, cfg)
    recognized = cky.is_grammatical(bad_tokens)
    single = build_tree(bad_table, 0, len(bad_tokens), cfg.startsymbol)
    multiple = build_tree_full(bad_table, 0, len(bad_tokens), cfg.startsymbol)
    assert recognized is False
    assert single is None
    assert multiple == []
    print(f'{label}: recognized={recognized}, one tree={single}, all trees={multiple}')


### Ambiguity Test

Construct a separate grammar using `io.StringIO`: start symbol `S`, rules `S -> S S` and `S -> a`. The sentence `a a a` has exactly two different trees. Both must be returned by `build_tree_full`; each must have the same three leaves in the input order.


In [ ]:
# 5. Ambiguous grammar: exactly two distinct trees for three tokens.
ambiguous_cfg = CFG(io.StringIO('S\nS -> S S\nS -> a\n'))
ambiguous_tokens = 'a a a'.split()
ambiguous_table = build_backpointer_table(ambiguous_tokens, ambiguous_cfg)
ambiguous_trees = build_tree_full(ambiguous_table, 0, len(ambiguous_tokens), 'S')

assert CKY(ambiguous_cfg).is_grammatical(ambiguous_tokens) is True
assert_tree_set(ambiguous_trees, ambiguous_cfg, ambiguous_tokens)
assert len(ambiguous_trees) == 2
assert build_tree(ambiguous_table, 0, len(ambiguous_tokens), 'S') in ambiguous_trees
print('Ambiguous sentence: a a a')
print('Number of distinct trees:', len(ambiguous_trees))
pprint(ambiguous_trees)


### Add Your Own Test

Add at least one further test, state the expected result, and check it with assertions. For example, use a longer ambiguous sentence, a grammar with ambiguity inside both child spans, or repeated grammar rules. Keep its output in your submission.


In [ ]:
# State the expected result and add your own assertions.
### START YOUR CODE ###
raise NotImplementedError("Complete your additional test before running the tests.")
### END YOUR CODE ###


---

### Submission Check

Restart the kernel, run every cell in order, and save the actual outputs. Submit `学号_姓名_w7_cfg.zip` on **Blackboard** by the assignment deadline, containing `学号_姓名_w7_cfg.ipynb` with T1–T6, plus any extra modules or custom grammar files needed to run your code.

You do not need to include the original `atis3.cfg`, an environment directory, caches or answer files. See [README_en.md](README_en.md) for details.
